# Week 14: AI Features with the Claude API

**Instructions:** Fill in the empty code cells. Press `Shift+Enter` to run.


## 1. Basic API Call

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY env var

msg = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=256,
    messages=[{'role': 'user', 'content': 'What is PM2.5 in 2 sentences?'}]
)
print(msg.content[0].text)
print(f'Tokens: {msg.usage.input_tokens} in / {msg.usage.output_tokens} out')


## 2. Data Summary with System Prompt

In [ ]:
import pandas as pd, anthropic

client = anthropic.Anthropic()
df = pd.DataFrame({'county':['Taipei','Taichung','Kaohsiung'],
                   'aqi':[52, 87, 103], 'pm25':[12.3, 28.7, 35.1]})
summary = df.to_string(index=False)

msg = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=300,
    system='You explain air quality data to the general public. Be brief and clear.',
    messages=[{'role': 'user', 'content': f'Air quality data:\n{summary}\n\nWhat should people know?'}]
)
print(msg.content[0].text)


## 3. Streamlit Chat Template

In [ ]:
# chat_app.py
chat_code = '''
import streamlit as st
import anthropic

st.title("Data Q&A Assistant")
st.caption("AI assistant — not a certified expert.")

client = anthropic.Anthropic()
if "messages" not in st.session_state:
    st.session_state.messages = []

for m in st.session_state.messages:
    with st.chat_message(m["role"]):
        st.write(m["content"])

if prompt := st.chat_input("Ask about the data..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"): st.write(prompt)
    with st.chat_message("assistant"):
        r = client.messages.create(
            model="claude-haiku-4-5-20251001", max_tokens=400,
            system="Help users understand Taiwan open data. Be concise.",
            messages=st.session_state.messages)
        reply = r.content[0].text
        st.write(reply)
    st.session_state.messages.append({"role": "assistant", "content": reply})
'''
print(chat_code)


## 4. API Key Safety

In [ ]:
import os
# NEVER hardcode your API key!
# Option 1: .env file  ->  from dotenv import load_dotenv; load_dotenv()
# Option 2: .streamlit/secrets.toml  ->  st.secrets['ANTHROPIC_API_KEY']
# Option 3: environment variable
key = os.environ.get('ANTHROPIC_API_KEY')
print(f'Key found: {key[:8]}...' if key else 'WARNING: key not set')
